# Initialization

In [1]:
!unzip pekan-raya-statistika.zip

Archive:  pekan-raya-statistika.zip
  inflating: Test/claude.csv         
  inflating: Test/deepseek.csv       
  inflating: Test/gemini.csv         
  inflating: Test/gpt.csv            
  inflating: Test/grok.csv           
  inflating: Test/perplexity.csv     
  inflating: Train/claude.csv        
  inflating: Train/deepseek.csv      
  inflating: Train/gemini.csv        
  inflating: Train/gpt.csv           
  inflating: Train/grok.csv          
  inflating: Train/perplexity.csv    
  inflating: sample_submission.csv   


In [2]:
!pip install emoji -q
!pip install catboost -q
!pip install deep-translator -q
!pip install langid -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 39.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


# Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import emoji
from deep_translator import GoogleTranslator
import langid
from catboost import CatBoostClassifier
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from sklearn.preprocessing import LabelEncoder
import torch
from sklearn.utils.class_weight import compute_class_weight
import re
import unicodedata
tqdm.pandas()

# Read Dataset

In [4]:
train_dir = 'Train/'
csv_files = [f for f in os.listdir(train_dir) if f.endswith('.csv')]

train_data = {}
for file in csv_files:
    file_path = os.path.join(train_dir, file)
    df_name = os.path.splitext(file)[0]
    train_data[df_name] = pd.read_csv(file_path)
    train_data[df_name]['AI'] = df_name

In [5]:
test_dir = 'Test/'
csv_files = [f for f in os.listdir(test_dir) if f.endswith('.csv')]

test_data = {}
for file in csv_files:
    file_path = os.path.join(test_dir, file)
    df_name = os.path.splitext(file)[0]

    df = pd.read_csv(file_path)
    df["CommentId"] = df["CommentId"].astype(str).apply(lambda x: f"{df_name}_{x}")
    df["AI"] = df_name
    test_data[df_name] = df

In [6]:
prefix_order = ['gpt', 'claude', 'deepseek', 'gemini', 'grok', 'perplexity']

combined_train = pd.concat(
    [train_data[prefix] for prefix in prefix_order if prefix in train_data],
    ignore_index=True
)
combined_train.head()

,Comment,At,AppVersion,Sentiment,AI
0,অনেক ভালো একটি এপ আপনার' ও ব্যবহার করুনঃ অনেক ...,2025-06-04 09:50:35,1.2025.147,2,gpt
1,very good at Al and useful,2025-06-16 01:40:52,1.2025.154,2,gpt
2,I love it,2025-06-21 05:59:25,1.2025.154,2,gpt
3,good app,2025-07-02 15:52:41,1.2025.175,2,gpt
4,improve,2025-04-06 09:05:10,1.2025.084,2,gpt


In [7]:
combined_test = pd.concat(test_data.values(), ignore_index=True)
combined_test.head()

,CommentId,Comment,At,AppVersion,AI
0,perplexity_1,perfect,2025-02-16 10:24:29,2.40.1,perplexity
1,perplexity_2,nice,2025-05-30 03:06:31,2.44.1,perplexity
2,perplexity_3,great work Perplexity Team,2025-02-08 02:32:34,2.39.0,perplexity
3,perplexity_4,তেমন একটা ভালো না টাকা চাই শুধু,2025-06-29 08:24:35,2.48.2,perplexity
4,perplexity_5,too good,2025-05-21 09:04:46,2.44.1,perplexity


In [8]:
prefix_order = ['gpt', 'claude', 'deepseek', 'gemini', 'grok', 'perplexity']
combined_test['prefix'] = combined_test['CommentId'].apply(lambda x: x.split('_')[0])
combined_test['comment_number'] = combined_test['CommentId'].apply(lambda x: int(x.split('_')[1]))
combined_test['prefix'] = pd.Categorical(combined_test['prefix'], categories=prefix_order, ordered=True)
combined_test = combined_test.sort_values(by=['prefix', 'comment_number']).drop(columns=['prefix', 'comment_number'])
combined_test.head()

,CommentId,Comment,At,AppVersion,AI
21491,gpt_1,super bussniss idea 💡 for chat gpt birrlant ap,2025-04-07 15:27:56,1.2025.084,gpt
21492,gpt_2,good,2025-04-30 14:54:05,1.2025.105,gpt
21493,gpt_3,Very nice,2025-04-26 06:44:56,1.2025.105,gpt
21494,gpt_4,could be more free,2025-06-07 22:44:57,1.2025.147,gpt
21495,gpt_5,very good very good very Nice,2025-04-16 08:30:34,1.2025.091,gpt


# Feature Engineering

We detected the language of each comment in the dataset and translated all non-English text into English. By unifying the text, we reduce variability caused by multilingual inputs and make the data more consistent for downstream tasks such as sentiment analysis. Unfortunately, some languages could not be translated due to tool limitations, so those comments remain in their original form.

In [ ]:
def detect_and_translate(text):
    try:
        lang, _ = langid.classify(text)
    except:
        return text, "unknown"

    if lang != "en":
        try:
            text = GoogleTranslator(source="auto", target="en").translate(text)
        except:
            pass
    return text, lang

In [ ]:
train = combined_train.copy()
train[["Comment_translated", "language"]] = combined_train["Comment"].progress_apply(
    lambda x: detect_and_translate(x) if isinstance(x, str) else (x, "unknown")
).apply(pd.Series)

In [ ]:
test = combined_test.copy()
test[["Comment_translated", "language"]] = combined_test["Comment"].progress_apply(
    lambda x: detect_and_translate(x) if isinstance(x, str) else (x, "unknown")
).apply(pd.Series)

AppVersion preprocessing

- Fill missing values using forward-fill and backward-fill within each AI group.

- Convert version strings into numeric tuples (major, minor, patch) for proper ordering.

- Assign ordinal encoding per AI (older versions → smaller numbers, newer versions → larger numbers).

- Handle unseen versions by assigning them the index of the latest version.

Comment preprocessing

- Handle missing comments by dropping or replacing with defaults.

- Use the translated comment if available, otherwise fall back to the original.

- Convert emojis into descriptive text (e.g., 😀 → :grinning_face:).

In [ ]:
train = pd.read_csv('new_train_detect_and_translate.csv')
test = pd.read_csv('new_test_detect_and_translate.csv')

In [ ]:
def parse_version(v):
    if pd.isna(v):
        return (0, 0, 0)
    parts = v.split(".")
    clean_parts = []
    for p in parts:
        m = re.match(r"(\d+)", p)
        if m:
            clean_parts.append(int(m.group(1)))
        else:
            clean_parts.append(0)
    clean_parts = clean_parts + [0]*(3-len(clean_parts))
    return tuple(clean_parts[:3])

#train
filled = (
    train.groupby("AI", group_keys=False)
         .apply(lambda g: g.sort_values("At")["AppVersion"].ffill().bfill())
         .reindex(train.index)   # balikin ke urutan asli
)

train["AppVersion"] = filled

# buat tuple versi
train["Version_tuple"] = train["AppVersion"].apply(parse_version)

mapping = {}

for ai, subdf in train.groupby("AI"):
    # unique version -> rank
    rank = (
        subdf["Version_tuple"]
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
        .reset_index()
        .set_index("Version_tuple")["index"]
    )
    mapping[ai] = rank.to_dict()

def encode_version(ai, version):
    if ai in mapping and version in mapping[ai]:
        return mapping[ai][version]
    else:
        return len(mapping[ai])  # unseen version diberi asumsi versi terbaru

train["AppVersion"] = train.apply(
    lambda r: encode_version(r["AI"], r["Version_tuple"]), axis=1
)

train = train.dropna(subset=['Comment']).reset_index(drop=True)
train['Comment_translated'] = train['Comment_translated'].fillna(train['Comment'])
train['Comment_translated'] = train['Comment_translated'].apply(lambda x: emoji.demojize(x))
train.drop(columns=['Comment', 'language','Version_tuple'],inplace=True)
train.rename(columns={'Comment_translated': 'Comment'}, inplace=True)

# test
filled = (
    test.groupby("AI", group_keys=False)
         .apply(lambda g: g.sort_values("At")["AppVersion"].ffill().bfill())
         .reindex(test.index)   # balikin ke urutan asli
)

test["AppVersion"] = filled
# buat tuple versi
test["Version_tuple"] = test["AppVersion"].apply(parse_version)

test["AppVersion"] = test.apply(
    lambda r: encode_version(r["AI"], r["Version_tuple"]), axis=1
)

test['Comment'] = test['Comment'].fillna('Good')
test['Comment_translated'] = test['Comment_translated'].fillna(test['Comment'])
test['Comment_translated'] = test['Comment_translated'].apply(lambda x: emoji.demojize(x))
test.drop(columns=['Comment', 'language','Version_tuple'],inplace=True)
test.rename(columns={'Comment_translated': 'Comment'}, inplace=True)

/tmp/ipython-input-1261442904.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sort_values("At")["AppVersion"].ffill().bfill())
/tmp/ipython-input-1261442904.py:60: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sort_values("At")["AppVersion"].ffill().bfill())


During data cleaning, we found that the dataset contained a surprisingly large number of duplicate rows. To ensure data quality and prevent the model from being biased by repeated samples, we removed these duplicates.

In [ ]:
# Find duplicate comments
duplicated_comments_mask = train.duplicated(subset=['Comment'], keep=False)
duplicated_comments_df = train[duplicated_comments_mask]

# Find unique comments
unique_comments_df = train[~duplicated_comments_mask].copy()

if not duplicated_comments_df.empty:
    most_frequent_sentiment = duplicated_comments_df.groupby('Comment')['Sentiment'].agg(lambda x: x.mode()[0]).reset_index()

    merged_df = pd.merge(most_frequent_sentiment, duplicated_comments_df, on=['Comment', 'Sentiment'], how='left')
    cleaned_duplicated_df = merged_df.drop_duplicates(subset=['Comment']).copy()

    train_cleaned = pd.concat([unique_comments_df, cleaned_duplicated_df], ignore_index=True)
else:
    train_cleaned = train.copy()

print(f"Original number of rows: {len(train)}")
print(f"Number of rows after removing duplicate comments based on most frequent sentiment: {len(train_cleaned)}")

train = train_cleaned

duplicate_comments_after = train.duplicated(subset=['Comment']).sum()
print(f"Number of duplicate comments after cleaning: {duplicate_comments_after}")

Original number of rows: 130880
Number of rows after removing duplicate comments based on most frequent sentiment: 77813
Number of duplicate comments after cleaning: 0


Some comments contained unusual or non-standard characters (e.g. special symbols, or visually similar Unicode variants). To standardize them, we applied Unicode NFKC normalization.

In [ ]:
def normalize_text(text: str) -> str:
    # 1. NFKC normalization
    text = unicodedata.normalize("NFKC", text)

    text = re.sub(r"[\u200B-\u200F\u202A-\u202E\u2060-\u206F]", "", text)

    text = re.sub(r"\s+", " ", text)

    text = text.strip().lower()

    return text

train['Comment'] = train['Comment'].apply(normalize_text)
test['Comment'] = test['Comment'].apply(normalize_text)

To represent each comment in a numerical form suitable for machine learning, we embedded the Comment column using a pretrained language model.

In [ ]:
train

,At,AppVersion,Sentiment,AI,Comment
0,2025-06-04 09:50:35,60,2,gpt,"a lot of good apps you 'o use: a lot of fun, a..."
1,2025-06-16 01:40:52,61,2,gpt,very good at al and useful
2,2025-06-17 13:49:37,62,2,gpt,this app has literally turned to be my best fr...
3,2025-05-12 14:16:10,57,2,gpt,freat app you can use it for anething
4,2025-07-03 14:20:45,64,2,gpt,very good and productive app
...,...,...,...,...,...
77808,2025-05-22 17:49:48,52,2,claude,you are the best
77809,2025-05-10 14:50:17,54,2,gpt,you the best
77810,2025-05-24 14:16:33,55,2,gpt,موريی .ه
77811,2025-04-09 13:22:11,51,2,gpt,★★★★★


In [ ]:
# Ubah tipe data AppVersion ke object
train['AppVersion'] = train['AppVersion'].astype(str)
test['AppVersion'] = test['AppVersion'].astype(str)

In [ ]:
train_df = train.copy()
test_df = test.copy()

print("Memuat semua model... (hanya sekali)")
embedding_model = SentenceTransformer('intfloat/multilingual-e5-large')
print("Semua model dimuat.")
print("\n" + "="*50 + "\n")

def create_features(df, embedding_model):

    # --- Embedding Text ---
    text_embeddings = embedding_model.encode(df['Comment'].tolist(), show_progress_bar=True)
    embedding_df = pd.DataFrame(text_embeddings, columns=[f'embed_{i}' for i in range(text_embeddings.shape[1])])

    return embedding_df

train_df = create_features(train_df, embedding_model)
test_df = create_features(test_df, embedding_model)

Memuat semua model... (hanya sekali)
Semua model dimuat.




Batches:   0%|          | 0/2432 [00:00<?, ?it/s]

Batches:   0%|          | 0/1437 [00:00<?, ?it/s]

In [ ]:
train_df['Sentiment'] = train['Sentiment']

# Modeling

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import numpy as np

# =====================================
# 1. Split Data (embedding sudah ada)
# =====================================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df.drop(columns=['Sentiment']),
    train_df["Sentiment"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=train_df["Sentiment"]
)

X_train = train_texts
X_val = val_texts
y_train = train_labels
y_val = val_labels

# =====================================
# 2. Hitung class weight
# =====================================
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", {i: w for i, w in zip(classes, class_weights)})

# =====================================
# 3. Dataset & DataLoader
# =====================================
train_dataset = TensorDataset(torch.tensor(X_train.values, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.long))
val_dataset   = TensorDataset(torch.tensor(X_val.values, dtype=torch.float32),
                              torch.tensor(y_val, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)

# =====================================
# 4. ANN Model
# =====================================
class ANNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_classes=3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.layers(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[1]  # dimensi embedding
num_classes = len(classes)
model = ANNClassifier(input_dim, hidden_dim=256, num_classes=num_classes).to(device)

# =====================================
# 5. Training Setup
# =====================================
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

# =====================================
# 6. Training Loop
# =====================================
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # --- Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_loss:.4f}")
    print(classification_report(all_labels, all_preds, target_names=["Negative","Neutral","Positive"]))

Class weights: {np.int64(0): np.float64(1.755350647153371), np.int64(1): np.float64(6.818928688793953), np.int64(2): np.float64(0.4378930485797493)}
Epoch 1/10, Train Loss: 0.7602
              precision    recall  f1-score   support

    Negative       0.76      0.87      0.81      2955
     Neutral       0.18      0.29      0.23       761
    Positive       0.96      0.89      0.93     11847

    accuracy                           0.86     15563
   macro avg       0.63      0.68      0.65     15563
weighted avg       0.89      0.86      0.87     15563

Epoch 2/10, Train Loss: 0.6702
              precision    recall  f1-score   support

    Negative       0.80      0.77      0.79      2955
     Neutral       0.19      0.50      0.28       761
    Positive       0.97      0.88      0.92     11847

    accuracy                           0.84     15563
   macro avg       0.65      0.72      0.66     15563
weighted avg       0.90      0.84      0.86     15563

Epoch 3/10, Train Loss: 0.6

Train with all dataset

In [ ]:
import torch.optim as optim

# =====================================
# 2. Split Data (Train/Val)
# =====================================
X = train_df.drop(columns=['Sentiment']).values
y = train_df['Sentiment'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Class weights
classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("Class weights:", dict(zip(classes, class_weights)))

# Convert ke TensorDataset
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.long))
val_dataset   = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                              torch.tensor(y_val, dtype=torch.long))
test_dataset  = TensorDataset(torch.tensor(test_df.values, dtype=torch.float32))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

# =====================================
# 3. ANN Model
# =====================================
class ANNClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(ANNClassifier, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.layers(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ANNClassifier(input_dim=X.shape[1], num_classes=len(classes)).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# =====================================
# 4. Training dengan Train/Val
# =====================================
EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            outputs = model(xb)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {total_loss:.4f}, Val Acc: {acc:.4f}")

# Report di val
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        outputs = model(xb)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(yb.numpy())
print("\nClassification Report (Validation):")
print(classification_report(all_true, all_preds, target_names=["Negative","Neutral","Positive"]))


# =====================================
# 5. Retrain di Full Train Data
# =====================================
print("\nRetrain di full training data...")
model_full = ANNClassifier(input_dim=X.shape[1], num_classes=len(classes)).to(device)
criterion_full = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer_full = optim.Adam(model_full.parameters(), lr=1e-4)

full_dataset = TensorDataset(torch.tensor(X, dtype=torch.float32),
                             torch.tensor(y, dtype=torch.long))
full_loader = DataLoader(full_dataset, batch_size=64, shuffle=True)

EPOCHS_FULL = 10
for epoch in range(EPOCHS_FULL):
    model_full.train()
    total_loss = 0
    for xb, yb in full_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer_full.zero_grad()
        outputs = model_full(xb)
        loss = criterion_full(outputs, yb)
        loss.backward()
        optimizer_full.step()
        total_loss += loss.item()
    print(f"[FULL TRAIN] Epoch {epoch+1}/{EPOCHS_FULL}, Loss: {total_loss:.4f}")


# =====================================
# 6. Inference di Test
# =====================================
print("\nInference di test set...")
model_full.eval()
y_test_pred = []
with torch.no_grad():
    for xb in test_loader:
        xb = xb[0].to(device)
        outputs = model_full(xb)
        preds = torch.argmax(outputs, dim=1)
        y_test_pred.extend(preds.cpu().numpy())

test_df["Predict"] = np.array(y_test_pred, dtype=int)
print(test_df.head())

Class weights: {np.int64(0): np.float64(1.755350647153371), np.int64(1): np.float64(6.818928688793953), np.int64(2): np.float64(0.4378930485797493)}
Epoch 1/10, Train Loss: 788.1466, Val Acc: 0.8194
Epoch 2/10, Train Loss: 683.7094, Val Acc: 0.8445
Epoch 3/10, Train Loss: 653.5593, Val Acc: 0.8254
Epoch 4/10, Train Loss: 640.7555, Val Acc: 0.8322
Epoch 5/10, Train Loss: 634.9325, Val Acc: 0.8008
Epoch 6/10, Train Loss: 625.8266, Val Acc: 0.8502
Epoch 7/10, Train Loss: 625.8397, Val Acc: 0.8077
Epoch 8/10, Train Loss: 620.0856, Val Acc: 0.8440
Epoch 9/10, Train Loss: 618.3573, Val Acc: 0.8440
Epoch 10/10, Train Loss: 615.3046, Val Acc: 0.8322

Classification Report (Validation):
              precision    recall  f1-score   support

    Negative       0.82      0.74      0.78      2955
     Neutral       0.19      0.55      0.29       761
    Positive       0.97      0.87      0.92     11847

    accuracy                           0.83     15563
   macro avg       0.66      0.72      0.

## Submission

In [ ]:
# --- Make Submission ---
test_df['CommentId'] = test['CommentId']
submission = test_df[["CommentId", "Predict"]].copy()
submission = submission.rename(columns={"Predict": "Sentiment"})

# --- Save CSV ---
submission.to_csv("submission ANN.csv", index=False)

submission.head()

,CommentId,Sentiment
0,gpt_1,2
1,gpt_2,2
2,gpt_3,2
3,gpt_4,1
4,gpt_5,2


In [ ]:
best = pd.read_csv('submission_new_teknik.csv')
print(best['Sentiment'].value_counts())
print(submission['Sentiment'].value_counts())

Sentiment
2    38083
0     5223
1     2665
Name: count, dtype: int64


In [ ]:
test_df['Predict'].value_counts()

,count
Predict,
2,37504
0,5116
1,3351
